In [ ]:

your_api_key = "your_api_key_here"
your_crn = "your_crn_here"

from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token=your_api_key,
    instance=your_crn,
    name="my_account",
    overwrite=True
)

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import matplotlib
import os
import math

# Try to set an interactive backend if available, else use Agg for saving
try:
    matplotlib.use('TkAgg')  # Try interactive backend (TkAgg for GUI environments)
except:
    matplotlib.use('Agg')  # Fallback to non-interactive for saving

# ---------- IBM Quantum Credentials ----------
API_TOKEN = "your_api_key_here"
CRN = "your_crn_here"

# ---------- Grover Components ----------
def grover_oracle(n, marked_state='11'):
    qc = QuantumCircuit(n)
    for i, bit in enumerate(marked_state[::-1]):
        if bit == '0':
            qc.x(i)
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    for i, bit in enumerate(marked_state[::-1]):
        if bit == '0':
            qc.x(i)
    return qc

def diffuser(n):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

def grover_circuit(n, marked_state, iterations):
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    oracle = grover_oracle(n, marked_state)
    diff = diffuser(n)
    for _ in range(iterations):
        qc.append(oracle, range(n))
        qc.append(diff, range(n))
    qc.measure(range(n), range(n))
    return qc

# ---------- Utility ----------
def counts_from_result(result):
    """Extract counts from simulator or IBM result."""
    if hasattr(result, 'get_counts'):
        return result.get_counts()
    else:
        # For SamplerV2 results
        data_bin = result[0].data
        if hasattr(data_bin, 'c'):  # Classical register named 'c' from measure
            return data_bin.c.get_counts()
        elif hasattr(data_bin, 'meas'):
            return data_bin.meas.get_counts()
        else:
            raise ValueError("No valid measurement data found in result")

# ---------- Runner for Local ----------
def run_grover_local(n=2, marked_state='11', iterations=1, shots=2048):
    qc = grover_circuit(n, marked_state, iterations)
    
    # ----- Ideal simulator -----
    sim_ideal = AerSimulator()
    circ_ideal = transpile(qc, sim_ideal)
    result_ideal = sim_ideal.run(circ_ideal, shots=shots).result()
    counts_ideal = counts_from_result(result_ideal)
    
    # ----- Noisy local simulator -----
    noise_model = NoiseModel()
    p1, p2 = 0.01, 0.02
    error_1q = depolarizing_error(p1, 1)
    error_2q = depolarizing_error(p2, 2)
    error_therm = thermal_relaxation_error(t1=100e-6, t2=80e-6, time=50e-9)
    error_1q_combined = error_1q.compose(error_therm)
    noise_model.add_all_qubit_quantum_error(error_1q_combined, ["h", "x", "y", "z"])
    noise_model.add_all_qubit_quantum_error(error_2q, ["cx"])
    sim_noisy = AerSimulator(noise_model=noise_model, seed_simulator=123)
    circ_noisy = transpile(qc, sim_noisy)
    result_noisy = sim_noisy.run(circ_noisy, shots=shots).result()
    counts_noisy = counts_from_result(result_noisy)
    
    return counts_ideal, counts_noisy

# ---------- Runner for IBM ----------
def run_grover_ibm(n=2, marked_state='11', iterations=1, shots=2048, backend_name=None):
    # Initialize service with provided credentials
    try:
        service = QiskitRuntimeService(channel="ibm_cloud", token=API_TOKEN, instance=CRN)
    except Exception as e:
        raise Exception(f"Failed to initialize QiskitRuntimeService: {str(e)}. Verify API token and CRN.")
    
    qc = grover_circuit(n, marked_state, iterations)
    
    if backend_name is None:
        raise ValueError("Backend name must be specified (e.g., 'ibm_brisbane').")
    
    try:
        backend = service.backend(backend_name)
    except Exception as e:
        raise Exception(f"Failed to access backend {backend_name}: {str(e)}. Check backend availability with service.backends().")
    
    circ = transpile(qc, backend)
    
    # Use Runtime SamplerV2 with mode=backend
    sampler = SamplerV2(mode=backend)
    job = sampler.run([circ], shots=shots)
    print(f"Job submitted to {backend_name}, Job ID: {job.job_id()}")
    result = job.result()
    counts = counts_from_result(result)
    
    return counts, backend_name

# ---------- Plot Function ----------
def plot_and_save_histograms(counts_dict, labels, filename_prefix='grover'):
    """Plot multiple histograms and save to file."""
    try:
        fig = plot_histogram(list(counts_dict.values()), legend=labels)
        output_path = os.path.abspath(f'{filename_prefix}_histogram.png')
        plt.savefig(output_path, bbox_inches='tight')
        print(f"Histogram saved to: {output_path}")
        
        # Attempt to display if interactive
        if matplotlib.get_backend().lower() not in ['agg', 'inline']:
            plt.show()
        plt.close(fig)
    except Exception as e:
        print(f"Error generating histogram: {str(e)}")
        print("Ensure matplotlib and qiskit[visualization] are installed (`pip install matplotlib qiskit[visualization]`) and check display settings.")

# ---------- Main Runner ----------
def run_grover_full(n=2, marked_state='11', shots=2048):
    # Calculate optimal iterations
    N = 2**n
    optimal_iterations = math.floor(math.pi / 4 * math.sqrt(N))
    print(f"Calculated optimal iterations for n={n} (N={N}): {optimal_iterations}")
    iteration_range = [optimal_iterations, optimal_iterations + 1]  # Test 1 and 2
    
    # Store results for all iterations
    all_results = {
        'local_ideal': {},
        'local_noisy': {},
        'ibm_brisbane': {},
        'ibm_torino': {}
    }
    
    # Run local simulations for each iteration count
    for it in iteration_range:
        print(f"\nRunning Grover's Algorithm on Local Simulators (iterations={it})...")
        counts_ideal, counts_noisy = run_grover_local(n, marked_state, it, shots)
        all_results['local_ideal'][it] = counts_ideal
        all_results['local_noisy'][it] = counts_noisy
        print(f"Ideal (Local, iter={it}):", counts_ideal)
        print(f"Noisy (Local, iter={it}):", counts_noisy)
        
        # Plot local results for this iteration
        plot_and_save_histograms(
            {'ideal_local': counts_ideal, 'noisy_local': counts_noisy},
            [f'ideal_local_iter{it}', f'noisy_local_iter{it}'],
            f'local_grover_iter{it}'
        )
    
    # Skip IBM Cloud Simulator (retired in 2024; use local ideal as proxy)
    print("\nSkipping IBM Cloud Simulator (retired in 2024; using local ideal instead)...")
    all_results['ibm_sim'] = all_results['local_ideal']  # Proxy
    sim_backend = "local_aer (proxy for retired IBM simulator)"
    print(f"IBM Simulator Proxy ({sim_backend}):", {it: counts for it, counts in all_results['ibm_sim'].items()})
    
    # Plot simulator proxy for each iteration
    for it in iteration_range:
        plot_and_save_histograms(
            {'ideal_local': all_results['local_ideal'][it], 'ibm_sim': all_results['ibm_sim'][it]},
            [f'ideal_local_iter{it}', f'ibm_sim_proxy_iter{it}'],
            f'ibm_sim_grover_iter{it}'
        )
    
    # Run on IBM hardware (ibm_brisbane and ibm_torino) for each iteration
    ibm_backends = ['ibm_brisbane', 'ibm_torino']
    for backend_name in ibm_backends:
        for it in iteration_range:
            print(f"\nRunning on IBM Hardware ({backend_name}, iterations={it})...")
            try:
                counts, hw_backend = run_grover_ibm(n, marked_state, it, shots, backend_name=backend_name)
                all_results[backend_name][it] = counts
                print(f"IBM Hardware ({hw_backend}, iter={it}):", counts)
                
                # Plot hardware vs ideal for this iteration
                plot_and_save_histograms(
                    {'ideal_local': all_results['local_ideal'][it], backend_name: counts},
                    [f'ideal_local_iter{it}', f'{backend_name}_iter{it}'],
                    f'{backend_name}_grover_iter{it}'
                )
            except Exception as e:
                print(f"Error running on IBM Hardware {backend_name} (iter={it}): {str(e)}")
                print("Hardware jobs may queue; check job status with job_id. Ensure backend is available with service.backends(simulator=False, operational=True).")
                all_results[backend_name][it] = None
    
    # Generate comparison plots for each iteration (ideal + both hardware)
    for it in iteration_range:
        all_counts = {'ideal_local': all_results['local_ideal'][it]}
        all_labels = [f'ideal_local_iter{it}']
        valid = True
        for backend_name in ibm_backends:
            if all_results[backend_name][it] is not None:
                all_counts[backend_name] = all_results[backend_name][it]
                all_labels.append(f'{backend_name}_iter{it}')
            else:
                valid = False
        if len(all_counts) > 1:  # Only plot if we have at least one hardware result
            plot_and_save_histograms(
                all_counts,
                all_labels,
                f'ibm_hw_comparison_grover_iter{it}'
            )
            print(f"\nComparison histogram (iter={it}) saved as 'ibm_hw_comparison_grover_iter{it}_histogram.png'")
        else:
            print(f"\nNo successful hardware runs for iter={it}; skipping comparison plot.")
    
    # Print recommendation
    print("\nOptimization Summary:")
    print(f"For n={n}, the theoretical optimal iterations is {optimal_iterations}.")
    print("Check the histograms to determine the best iteration count for each backend.")
    print("Look for the iteration where the marked state '11' has the highest probability.")
    
    return all_results

# ---------- Setup Instructions ----------
print("""
Running with provided IBM Cloud API token and CRN.
Dependencies required:
- pip install --upgrade qiskit qiskit-aer qiskit-ibm-runtime qiskit[visualization] matplotlib
Notes:
- IBM cloud simulators retired May 2024; using local AerSimulator as ideal proxy.
- IBM hardware (ibm_brisbane and ibm_torino) are Eagle/Heron processors; jobs may queue.
- If backends are unavailable, the code will print available ones.
- Use job_id to retrieve hardware results if queued (e.g., job = service.job('job_id'); result = job.result()).
- Testing iterations around theoretical optimum (1, 2) for n=2.
- Comparison plots generated for each iteration count if hardware runs succeed.
""")

# ---------- Run ----------
if __name__ == "__main__":
    run_grover_full(n=2, marked_state='11', shots=2048)


Running with provided IBM Cloud API token and CRN.
Dependencies required:
- pip install --upgrade qiskit qiskit-aer qiskit-ibm-runtime qiskit[visualization] matplotlib
Notes:
- IBM cloud simulators retired May 2024; using local AerSimulator as ideal proxy.
- IBM hardware (ibm_brisbane and ibm_torino) are Eagle/Heron processors; jobs may queue.
- If backends are unavailable, the code will print available ones.
- Use job_id to retrieve hardware results if queued (e.g., job = service.job('job_id'); result = job.result()).
- Testing iterations around theoretical optimum (1, 2) for n=2.
- Comparison plots generated for each iteration count if hardware runs succeed.

Calculated optimal iterations for n=2 (N=4): 1

Running Grover's Algorithm on Local Simulators (iterations=1)...
Ideal (Local, iter=1): {'11': 2048}
Noisy (Local, iter=1): {'01': 27, '10': 56, '00': 32, '11': 1933}
Histogram saved to: /Users/msu/cwqconda/qgss-2025 practice/Grover paper/local_grover_iter1_histogram.png

Running 

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import matplotlib
import os
import math

# Try to set an interactive backend if available, else use Agg for saving
try:
    matplotlib.use('TkAgg')  # Try interactive backend (TkAgg for GUI environments)
except:
    matplotlib.use('Agg')  # Fallback to non-interactive for saving

# ---------- IBM Quantum Credentials ----------
API_TOKEN = "your_api_key_here"
CRN = "your_crn_here"

# ---------- Grover Components ----------
def grover_oracle(n, marked_state='1111'):
    qc = QuantumCircuit(n)
    for i, bit in enumerate(marked_state[::-1]):
        if bit == '0':
            qc.x(i)
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    for i, bit in enumerate(marked_state[::-1]):
        if bit == '0':
            qc.x(i)
    return qc

def diffuser(n):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

def grover_circuit(n, marked_state, iterations):
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    oracle = grover_oracle(n, marked_state)
    diff = diffuser(n)
    for _ in range(iterations):
        qc.append(oracle, range(n))
        qc.append(diff, range(n))
    qc.measure(range(n), range(n))
    return qc

# ---------- Utility ----------
def counts_from_result(result):
    """Extract counts from simulator or IBM result."""
    if hasattr(result, 'get_counts'):
        return result.get_counts()
    else:
        # For SamplerV2 results
        data_bin = result[0].data
        if hasattr(data_bin, 'c'):  # Classical register named 'c' from measure
            return data_bin.c.get_counts()
        elif hasattr(data_bin, 'meas'):
            return data_bin.meas.get_counts()
        else:
            raise ValueError("No valid measurement data found in result")

# ---------- Runner for Local ----------
def run_grover_local(n=4, marked_state='1111', iterations=1, shots=2048):
    qc = grover_circuit(n, marked_state, iterations)
    
    # ----- Ideal simulator -----
    sim_ideal = AerSimulator()
    circ_ideal = transpile(qc, sim_ideal)
    result_ideal = sim_ideal.run(circ_ideal, shots=shots).result()
    counts_ideal = counts_from_result(result_ideal)
    
    # ----- Noisy local simulator -----
    noise_model = NoiseModel()
    p1, p2 = 0.01, 0.02
    error_1q = depolarizing_error(p1, 1)
    error_2q = depolarizing_error(p2, 2)
    error_therm = thermal_relaxation_error(t1=100e-6, t2=80e-6, time=50e-9)
    error_1q_combined = error_1q.compose(error_therm)
    noise_model.add_all_qubit_quantum_error(error_1q_combined, ["h", "x", "y", "z"])
    noise_model.add_all_qubit_quantum_error(error_2q, ["cx"])
    sim_noisy = AerSimulator(noise_model=noise_model, seed_simulator=123)
    circ_noisy = transpile(qc, sim_noisy)
    result_noisy = sim_noisy.run(circ_noisy, shots=shots).result()
    counts_noisy = counts_from_result(result_noisy)
    
    return counts_ideal, counts_noisy

# ---------- Runner for IBM ----------
def run_grover_ibm(n=4, marked_state='1111', iterations=1, shots=2048, backend_name=None):
    # Initialize service with provided credentials
    try:
        service = QiskitRuntimeService(channel="ibm_cloud", token=API_TOKEN, instance=CRN)
    except Exception as e:
        raise Exception(f"Failed to initialize QiskitRuntimeService: {str(e)}. Verify API token and CRN.")
    
    qc = grover_circuit(n, marked_state, iterations)
    
    if backend_name is None:
        raise ValueError("Backend name must be specified (e.g., 'ibm_brisbane').")
    
    try:
        backend = service.backend(backend_name)
    except Exception as e:
        raise Exception(f"Failed to access backend {backend_name}: {str(e)}. Check backend availability with service.backends().")
    
    circ = transpile(qc, backend)
    
    # Use Runtime SamplerV2 with mode=backend
    sampler = SamplerV2(mode=backend)
    job = sampler.run([circ], shots=shots)
    print(f"Job submitted to {backend_name}, Job ID: {job.job_id()}")
    result = job.result()
    counts = counts_from_result(result)
    
    return counts, backend_name

# ---------- Plot Function ----------
def plot_and_save_histograms(counts_dict, labels, filename_prefix='grover'):
    """Plot multiple histograms and save to file."""
    try:
        fig = plot_histogram(list(counts_dict.values()), legend=labels)
        output_path = os.path.abspath(f'{filename_prefix}_histogram.png')
        plt.savefig(output_path, bbox_inches='tight')
        print(f"Histogram saved to: {output_path}")
        
        # Attempt to display if interactive
        if matplotlib.get_backend().lower() not in ['agg', 'inline']:
            plt.show()
        plt.close(fig)
    except Exception as e:
        print(f"Error generating histogram: {str(e)}")
        print("Ensure matplotlib and qiskit[visualization] are installed (`pip install matplotlib qiskit[visualization]`) and check display settings.")

# ---------- Main Runner ----------
def run_grover_full(n=4, marked_state='1111', shots=2048):
    # Calculate optimal iterations
    N = 2**n
    optimal_iterations = math.floor(math.pi / 4 * math.sqrt(N))
    print(f"Calculated optimal iterations for n={n} (N={N}): {optimal_iterations}")
    iteration_range = [optimal_iterations - 1, optimal_iterations, optimal_iterations + 1]  # Test around optimal
    
    # Store results for all iterations
    all_results = {
        'local_ideal': {},
        'local_noisy': {},
        'ibm_brisbane': {},
        'ibm_torino': {}
    }
    
    # Run local simulations for each iteration count
    for it in iteration_range:
        print(f"\nRunning Grover's Algorithm on Local Simulators (iterations={it})...")
        counts_ideal, counts_noisy = run_grover_local(n, marked_state, it, shots)
        all_results['local_ideal'][it] = counts_ideal
        all_results['local_noisy'][it] = counts_noisy
        print(f"Ideal (Local, iter={it}):", counts_ideal)
        print(f"Noisy (Local, iter={it}):", counts_noisy)
        
        # Plot local results for this iteration
        plot_and_save_histograms(
            {'ideal_local': counts_ideal, 'noisy_local': counts_noisy},
            [f'ideal_local_iter{it}', f'noisy_local_iter{it}'],
            f'local_grover_iter{it}'
        )
    
    # Skip IBM Cloud Simulator (retired in 2024; use local ideal as proxy)
    print("\nSkipping IBM Cloud Simulator (retired in 2024; using local ideal instead)...")
    all_results['ibm_sim'] = all_results['local_ideal']  # Proxy
    sim_backend = "local_aer (proxy for retired IBM simulator)"
    print(f"IBM Simulator Proxy ({sim_backend}):", {it: counts for it, counts in all_results['ibm_sim'].items()})
    
    # Plot simulator proxy for each iteration
    for it in iteration_range:
        plot_and_save_histograms(
            {'ideal_local': all_results['local_ideal'][it], 'ibm_sim': all_results['ibm_sim'][it]},
            [f'ideal_local_iter{it}', f'ibm_sim_proxy_iter{it}'],
            f'ibm_sim_grover_iter{it}'
        )
    
    # Run on IBM hardware (ibm_brisbane and ibm_torino) for each iteration
    ibm_backends = ['ibm_brisbane', 'ibm_torino']
    for backend_name in ibm_backends:
        for it in iteration_range:
            print(f"\nRunning on IBM Hardware ({backend_name}, iterations={it})...")
            try:
                counts, hw_backend = run_grover_ibm(n, marked_state, it, shots, backend_name=backend_name)
                all_results[backend_name][it] = counts
                print(f"IBM Hardware ({hw_backend}, iter={it}):", counts)
                
                # Plot hardware vs ideal for this iteration
                plot_and_save_histograms(
                    {'ideal_local': all_results['local_ideal'][it], backend_name: counts},
                    [f'ideal_local_iter{it}', f'{backend_name}_iter{it}'],
                    f'{backend_name}_grover_iter{it}'
                )
            except Exception as e:
                print(f"Error running on IBM Hardware {backend_name} (iter={it}): {str(e)}")
                print("Hardware jobs may queue; check job status with job_id. Ensure backend is available with service.backends(simulator=False, operational=True).")
                all_results[backend_name][it] = None
    
    # Generate comparison plots for each iteration (ideal + both hardware)
    for it in iteration_range:
        all_counts = {'ideal_local': all_results['local_ideal'][it]}
        all_labels = [f'ideal_local_iter{it}']
        valid = True
        for backend_name in ibm_backends:
            if all_results[backend_name][it] is not None:
                all_counts[backend_name] = all_results[backend_name][it]
                all_labels.append(f'{backend_name}_iter{it}')
            else:
                valid = False
        if len(all_counts) > 1:  # Only plot if we have at least one hardware result
            plot_and_save_histograms(
                all_counts,
                all_labels,
                f'ibm_hw_comparison_grover_iter{it}'
            )
            print(f"\nComparison histogram (iter={it}) saved as 'ibm_hw_comparison_grover_iter{it}_histogram.png'")
        else:
            print(f"\nNo successful hardware runs for iter={it}; skipping comparison plot.")
    
    # Print recommendation
    print("\nOptimization Summary:")
    print(f"For n={n}, the theoretical optimal iterations is {optimal_iterations}.")
    print("Check the histograms to determine the best iteration count for each backend.")
    print("Look for the iteration where the marked state '1111' has the highest probability.")
    
    return all_results

# ---------- Setup Instructions ----------
print("""
Running with provided IBM Cloud API token and CRN.
Dependencies required:
- pip install --upgrade qiskit qiskit-aer qiskit-ibm-runtime qiskit[visualization] matplotlib
Notes:
- IBM cloud simulators retired May 2024; using local AerSimulator as ideal proxy.
- IBM hardware (ibm_brisbane and ibm_torino) are Eagle/Heron processors; jobs may queue.
- If backends are unavailable, the code will print available ones.
- Use job_id to retrieve hardware results if queued (e.g., job = service.job('job_id'); result = job.result()).
- Testing iterations around theoretical optimum (2, 3, 4) for n=4.
- Comparison plots generated for each iteration count if hardware runs succeed.
""")

# ---------- Run ----------
if __name__ == "__main__":
    run_grover_full(n=4, marked_state='1111', shots=2048)